# Modele de Reponse aux Promotions – AnyCompany Food & Beverage

**Objectif** : Predire l'impact d'une promotion sur les ventes (montant des ventes pendant les periodes promotionnelles)

**Modele** : GradientBoostingRegressor

**Donnees** : `ANYCOMPANY_LAB.ANALYTICS.VENTES_ENRICHIES` (transactions avec promo)

In [ ]:
%%sql -r df_sales
SELECT
    v.TRANSACTION_ID,
    v.AMOUNT,
    v.ANNEE,
    v.TRIMESTRE,
    v.MOIS,
    v.JOUR_SEMAINE,
    v.REGION,
    v.PAYMENT_METHOD,
    v.PENDANT_PROMO,
    COALESCE(v.DISCOUNT_PERCENTAGE, 0) AS DISCOUNT_PERCENTAGE,
    COALESCE(v.PROMOTION_TYPE, 'None') AS PROMOTION_TYPE,
    COALESCE(v.CAMPAIGN_TYPE, 'None') AS CAMPAIGN_TYPE,
    COALESCE(v.CAMPAIGN_BUDGET, 0) AS CAMPAIGN_BUDGET,
    COALESCE(v.CAMPAIGN_REACH, 0) AS CAMPAIGN_REACH,
    COALESCE(v.CAMPAIGN_CONVERSION_RATE, 0) AS CAMPAIGN_CONVERSION_RATE
FROM ANYCOMPANY_LAB.ANALYTICS.VENTES_ENRICHIES v
WHERE v.TRANSACTION_TYPE = 'Sale'

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

print(f"Shape: {df_sales.shape}")
print(f"\nVentes avec promo: {df_sales['PENDANT_PROMO'].sum()}")
print(f"Ventes sans promo: {(~df_sales['PENDANT_PROMO'].astype(bool)).sum()}")
print(f"\nMontant moyen avec promo: ${df_sales[df_sales['PENDANT_PROMO'].astype(bool)]['AMOUNT'].mean():,.2f}")
print(f"Montant moyen sans promo: ${df_sales[~df_sales['PENDANT_PROMO'].astype(bool)]['AMOUNT'].mean():,.2f}")

In [ ]:
df = df_sales.copy()

le_region = LabelEncoder()
le_payment = LabelEncoder()
le_promo_type = LabelEncoder()
le_camp_type = LabelEncoder()

df['REGION_ENC'] = le_region.fit_transform(df['REGION'].astype(str))
df['PAYMENT_ENC'] = le_payment.fit_transform(df['PAYMENT_METHOD'].astype(str))
df['PROMO_TYPE_ENC'] = le_promo_type.fit_transform(df['PROMOTION_TYPE'].astype(str))
df['CAMP_TYPE_ENC'] = le_camp_type.fit_transform(df['CAMPAIGN_TYPE'].astype(str))
df['PENDANT_PROMO_INT'] = df['PENDANT_PROMO'].astype(int)

feature_cols = ['ANNEE', 'TRIMESTRE', 'MOIS', 'JOUR_SEMAINE',
                'REGION_ENC', 'PAYMENT_ENC', 'PENDANT_PROMO_INT',
                'DISCOUNT_PERCENTAGE', 'PROMO_TYPE_ENC', 'CAMP_TYPE_ENC',
                'CAMPAIGN_BUDGET', 'CAMPAIGN_REACH', 'CAMPAIGN_CONVERSION_RATE']

X = df[feature_cols].fillna(0)
y = df['AMOUNT']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Montant moyen (train): ${y_train.mean():,.2f}")
print(f"Montant moyen (test): ${y_test.mean():,.2f}")

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    subsample=0.8
)

gbr.fit(X_train, y_train)

y_pred = gbr.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=" * 50)
print("PERFORMANCE DU MODELE")
print("=" * 50)
print(f"MAE  (erreur absolue moyenne): ${mae:,.2f}")
print(f"RMSE (racine erreur quadratique): ${rmse:,.2f}")
print(f"R2   (coefficient determination): {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_test, y_pred, alpha=0.3, s=10)
axes[0].plot([0, y_test.max()], [0, y_test.max()], 'r--')
axes[0].set_xlabel('Montant Reel')
axes[0].set_ylabel('Montant Predit')
axes[0].set_title('Reel vs Predit')

importances = pd.Series(gbr.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Importance des Features')

residuals = y_test.values - y_pred
axes[2].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[2].set_xlabel('Residus ($)')
axes[2].set_ylabel('Frequence')
axes[2].set_title('Distribution des Residus')

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 60)
print("ANALYSE DE L'IMPACT DES PROMOTIONS")
print("=" * 60)

X_with_promo = X_test.copy()
X_without_promo = X_test.copy()
X_with_promo['PENDANT_PROMO_INT'] = 1
X_with_promo['DISCOUNT_PERCENTAGE'] = 0.15
X_without_promo['PENDANT_PROMO_INT'] = 0
X_without_promo['DISCOUNT_PERCENTAGE'] = 0

pred_with = gbr.predict(X_with_promo)
pred_without = gbr.predict(X_without_promo)

lift = (pred_with.mean() - pred_without.mean()) / pred_without.mean() * 100

print(f"\nVente moyenne predite AVEC promo (15%): ${pred_with.mean():,.2f}")
print(f"Vente moyenne predite SANS promo:        ${pred_without.mean():,.2f}")
print(f"Lift promotionnel estime:                 {lift:+.2f}%")

print(f"\nTop 3 facteurs influencant le montant:")
for i, (feat, imp) in enumerate(importances.sort_values(ascending=False).head(3).items()):
    print(f"  {i+1}. {feat}: {imp:.4f}")

print(f"\nRecommandations:")
print(f"  - Ajuster les taux de remise selon la sensibilite par categorie")
print(f"  - Cibler les periodes et regions a plus fort impact")
print(f"  - Combiner promotions avec campagnes marketing pour maximiser le ROI")

In [ ]:
%%sql -r ctx
USE DATABASE ANYCOMPANY_LAB;
USE SCHEMA ANALYTICS;